# 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler

# 2. Config

In [ ]:
# Paths
CONTEXTS_JSON = Path("contexts.json")

SEED = 42
OUTPUT_DIR = Path("SynthCAER") / f"seed_{SEED}"

# Model
SDXL_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 9.0 
IMAGE_WIDTH = 1024
IMAGE_HEIGHT = 1024

# Number of context to generate images
MAX_CONTEXTS = None

# Replace created image
SKIP_EXISTING = False

# 3. Prompt config

In [ ]:
# Emotions. Keys from Schuller et al. 2026
EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]

EMOTION_MODIFIERS = {
    'Angry':    'furious, angry, enraged',
    'Disgust':  'disgusted, revolted, repulsed',
    'Fear':     'terrified, fearful, scared',
    'Happy':    'joyful, happy, cheerful',
    'Sad':      'sad, sorrowful, melancholic',
    'Surprise': 'surprised, astonished, shocked',
    'Neutral':  'neutral, calm, composed',
}

EMOTION_KEYS = {
    "Angry":    "anger and rage",
    "Disgust":  "disgust and loathing",
    "Fear":     "fear and terror",
    "Happy":    "happiness and joy",
    "Sad":      "sadness and grief",
    "Surprise": "surprise and amazement",
    "Neutral":  "neutral",
}

# Technic modifiers
QUALITY_SHORT   = "photorealistic, RAW photo, 8k, sharp focus"

NEGATIVE_PROMPT = (
    "blurry, low quality, bad anatomy, deformed face, extra limbs, "
    "cartoon, anime, painting, illustration, drawing, render, CGI, "
    "3D, digital art, text, watermark, unrealistic, plastic skin, "
    "no people, empty scene, "
    "back turned to camera, facing away from camera, "
    "person too small, tiny person, person in far background, "
    "hidden face, no face visible, covered face, silhouette, "
    "aerial view, bird eye view, extreme wide shot"
)

# Full prompt
def build_context_prompt(context, emotion):

    # Prompt 1: emotion + quality + person anchor
    prompt = (
        f"{EMOTION_MODIFIERS[emotion]}, {EMOTION_KEYS[emotion]}, "
        f"{QUALITY_SHORT}, "
        f"one person facing camera, face clearly visible, medium shot"
    )
    # Prompt_2 : emotion + context + quality + person anchor
    prompt_2 = (
        f"{EMOTION_MODIFIERS[emotion]}, {EMOTION_KEYS[emotion]}, "
        f"{QUALITY_SHORT}, "
        f"person prominently facing camera in foreground, "
        f"environmental portrait, recognizable {context} background"
    )
    return prompt, prompt_2

# 4. Categories load

Output from "syntcaer_categories.ipynb"

In [ ]:
with open(CONTEXTS_JSON, encoding="utf-8") as f:
    ctx_data = json.load(f)

all_contexts = ctx_data["categories"]

print(f"Contexts: {len(all_contexts)}")
print(f"Sources: {ctx_data["sources"]}")

# Sample categories
contexts = all_contexts[:MAX_CONTEXTS] if MAX_CONTEXTS is not None else all_contexts

# Ejecution summary
print(f"\nContexts: {len(contexts)}")
print(f"Total images: {len(contexts) * len(EMOTIONS)}")

# 5. StableDiffusion XL load

In [ ]:
# Check if cuda avaiable
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Dispositivo: {device}  |  dtype: {dtype}")

# Model pipeline
scheduler = EulerDiscreteScheduler.from_pretrained(SDXL_MODEL, subfolder="scheduler")

pipe = StableDiffusionXLPipeline.from_pretrained(
    SDXL_MODEL,
    scheduler=scheduler,
    torch_dtype=dtype,
    use_safetensors=True,
    variant="fp16" if device == "cuda" else None,
)
pipe = pipe.to(device)

if device == "cuda":
    pipe.enable_model_cpu_offload()
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pipe.enable_attention_slicing()
else:
    pipe.enable_attention_slicing()

print("SDXL loaded")

# 6. Image generation

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Generation loop
records = []
for ctx_idx, context in tqdm(enumerate(contexts), total=len(contexts), desc="Contexts generated:"):
    safe_name = context.replace(" ", "_").replace("/", "-")
    ctx_dir   = OUTPUT_DIR / f"{ctx_idx:04d}_{safe_name}"
    ctx_dir.mkdir(exist_ok=True)

    # Emotion loop
    for emotion in tqdm(EMOTIONS, desc="Emotions generated:", leave=False):
        # Path
        out_path = ctx_dir / f"{emotion}.png"

        # Prompt
        prompt_enc, prompt_2_enc = build_context_prompt(context, emotion)

        # No replace mode
        if SKIP_EXISTING and out_path.exists():
            records.append({
                "ctx_idx": ctx_idx, "context": context, "emotion": emotion,
                "emotion_prompt": prompt_enc, "scene_prompt": prompt_2_enc,
                "image_path": out_path.as_posix(),
            })
            continue

        # Generation
        result = pipe(
            prompt=prompt_enc,
            prompt_2=prompt_2_enc,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            width=IMAGE_WIDTH,
            height=IMAGE_HEIGHT,
            generator=torch.Generator(device=device).manual_seed(SEED),
        )
        result.images[0].save(out_path)

        # Index table
        records.append({
            "ctx_idx": ctx_idx, "context": context, "emotion": emotion,
            "emotion_prompt": prompt_enc, "scene_prompt": prompt_2_enc,
            "image_path": out_path.as_posix(),
        })

df_results = pd.DataFrame(records)
df_results.to_parquet(OUTPUT_DIR / "generation_results.parquet", index=False)
print(f"\n {len(records)} images generated")

# 7. Image grid per context

In [ ]:
EMOTION_COLORS = {
    "Angry":"#d62728",
    "Disgust":"#9467bd",
    "Fear":"#8c564b",
    "Happy":"#2ca02c",
    "Sad": "#1f77b4",
    "Surprise": "#ff7f0e",
    "Neutral": "#7f7f7f",
}

def plot_context_grid(df, ctx_idx):
    subset = df[df["ctx_idx"] == ctx_idx]

    context = subset.iloc[0]["context"]
    fig, axes = plt.subplots(1, len(EMOTIONS), figsize=(len(EMOTIONS) * 3, 3.8))

    for ax, emotion in zip(axes, EMOTIONS):
        row = subset[subset["emotion"] == emotion]
        ax.imshow(Image.open(row.iloc[0]["image_path"]))
        ax.set_title(emotion, fontsize=10, fontweight="bold", color=EMOTION_COLORS[emotion])
        ax.axis("off")

    fig.suptitle(f"Contexto: {context}", fontsize=10, fontweight="bold", y=1.03)
    plt.tight_layout()

    safe_name = context.replace(" ", "_").replace("/", "-")
    out = OUTPUT_DIR / f"grid_{ctx_idx:04d}_{safe_name}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Guardado: {out}")

    plt.close()

for idx in sorted(df_results["ctx_idx"].unique()):
    plot_context_grid(df_results, idx)